<a href="https://colab.research.google.com/github/Harry120705/HospitALL/blob/main/simuladorFilaRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação de Desempenho e Simulação de Eventos Discretos (SimPy)
## Fila de Atendimento de Caixa sob Condição Inicial de Sobrecarga ($L_q(0) = 44$)

---

### 1. Contexto do Experimento e Objetivo
Este notebook implementa um modelo estocástico de **Simulação de Eventos Discretos (DES)** com **Python** e **SimPy**, reproduzindo com rigor metodológico a operação de um caixa de atendimento no período crítico das **17:30 às 18:30** (3600 segundos).

O modelo fundamenta-se **ESTRITAMENTE** em três conjuntos de dados empíricos coletados em campo:
1. **`Prancheta 1 - Observador de Chegadas`**: Contagem de chegadas agrupadas em intervalos de 5 minutos, utilizada para calibrar a taxa de chegadas ($\lambda$).
2. **`Prancheta 2 - Observador de Atendimento`**: Amostras reais de duração do serviço no caixa em segundos, utilizada para extrair a distribuição de serviço ($\mu$).
3. **`Prancheta 3 - Rastreador de Espera`**: Rastreamento individual longitudinal de clientes (horário de entrada na fila, saída e tempo total), empregado na validação das métricas do modelo.

### 2. Condição de Contorno Crítica: Sobrecarga Inicial ($t = 0$)
Às 17:30 ($t = 0$), a fila **não inicia vazia**: existem exatamente **44 pessoas pré-acumuladas** aguardando atendimento. Essa condição introduz um regime transitório de alta saturação que demanda análise detalhada da taxa de esvaziamento (*burn-down*).



In [1]:
# Instalação das dependências necessárias
!pip install simpy plotly pandas numpy nbformat -q

import os
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import simpy
import plotly.express as px

# Configuração de semente global para reprodutibilidade estrita
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Resolução dinâmica de caminhos (local e Colab)
def resolver_caminho_arquivo(nome_arquivo):
    for p in [nome_arquivo, os.path.join('.', nome_arquivo), os.path.join('/content', nome_arquivo)]:
        if os.path.exists(p):
            return p
    return nome_arquivo

path_chegadas = resolver_caminho_arquivo('Prancheta 1 - Observador de Chegadas - Página1.csv')
path_atendimento = resolver_caminho_arquivo('Prancheta 2 - Observador de Atendimento - Página1.csv')
path_espera = resolver_caminho_arquivo('Prancheta 3 - Rastreador de Espera - Página1.csv')

print("Ambiente configurado com plotly.express.")


Ambiente configurado com plotly.express.


In [ ]:
def converter_para_segundos(t_str):
    """Converte strings de tempo nos formatos HH:MM:SS ou MM:SS para segundos inteiros."""
    if pd.isna(t_str):
        return np.nan
    t_str = str(t_str).strip()
    partes = t_str.split(':')
    try:
        if len(partes) == 2:
            return int(partes[0]) * 60 + int(partes[1])
        elif len(partes) == 3:
            return int(partes[0]) * 3600 + int(partes[1]) * 60 + int(partes[2])
        return float(t_str)
    except Exception:
        return np.nan

def carregar_e_tratar_dados(path_chegadas, path_atendimento, path_espera, usar_102_amostras=True):
    """
    Carrega e limpa os 3 arquivos CSV reais com validação de erros e tratamento robusto.
    """
    # ==========================================
    # 1. PRANCHETA 1: Observador de Chegadas
    # ==========================================
    try:
        df1_raw = pd.read_csv(path_chegadas, header=None)
        # Localiza dinamicamente a linha que contém o cabeçalho 'Horário'
        idx_cabecalho1 = df1_raw[df1_raw.iloc[:, 0].astype(str).str.contains('Horário', na=False)].index[0]
        df_chegadas = df1_raw.iloc[idx_cabecalho1 + 1:].copy()
        df_chegadas.columns = [str(c).strip() for c in df1_raw.iloc[idx_cabecalho1].values]
        df_chegadas['Total de Pessoas'] = pd.to_numeric(df_chegadas['Total de Pessoas'], errors='coerce').fillna(0).astype(int)

        total_chegadas = int(df_chegadas['Total de Pessoas'].sum())
        duracao_min = 60.0
        duracao_seg = 3600.0
        lambda_minuto = total_chegadas / duracao_min
        lambda_segundo = total_chegadas / duracao_seg
    except Exception as e:
        raise RuntimeError(f"Erro ao processar Prancheta 1 (Chegadas): {e}")

    # ==========================================
    # 2. PRANCHETA 2: Observador de Atendimento
    # ==========================================
    try:
        df2_raw = pd.read_csv(path_atendimento, header=None)
        # Localiza linha do cabeçalho 'Amostra'
        idx_cabecalho2 = df2_raw[df2_raw.iloc[:, 0].astype(str).str.contains('Amostra', na=False)].index[0]
        df2_dados = df2_raw.iloc[idx_cabecalho2 + 1:].copy()

        # Coluna 1 contém os tempos de atendimento em segundos
        amostras_todas = pd.to_numeric(df2_dados.iloc[:, 1], errors='coerce').dropna().values

        # O enunciado requer a utilização das 102 amostras válidas da Prancheta 2
        if usar_102_amostras and len(amostras_todas) >= 102:
            amostras_servico = amostras_todas[:102]
        else:
            amostras_servico = amostras_todas

        tempo_servico_medio = float(np.mean(amostras_servico))
        mu_estimado = 1.0 / tempo_servico_medio  # atendimentos por segundo
    except Exception as e:
        raise RuntimeError(f"Erro ao processar Prancheta 2 (Atendimento): {e}")

    # ==========================================
    # 3. PRANCHETA 3: Rastreador de Espera
    # ==========================================
    try:
        df3_raw = pd.read_csv(path_espera, header=None)
        idx_cabecalho3 = df3_raw[df3_raw.iloc[:, 0].astype(str).str.contains('Folha', na=False)].index[0]
        df_espera = df3_raw.iloc[idx_cabecalho3 + 1:].copy()
        df_espera.columns = ['folha', 'aluno', 'hora_chegada', 'hora_saida', 'tempo_fila_str']

        # Filtrar registros cancelados / riscados ou valores ausentes
        df_espera = df_espera.dropna(subset=['hora_chegada', 'hora_saida']).copy()
        df_espera = df_espera[~df_espera['aluno'].astype(str).str.contains('riscado', case=False, na=False)].copy()

        # Referência t0 = 17:30:00 (em segundos do dia)
        t0_segundos = 17 * 3600 + 30 * 60
        df_espera['chegada_seg'] = df_espera['hora_chegada'].apply(converter_para_segundos) - t0_segundos
        df_espera['saida_seg'] = df_espera['hora_saida'].apply(converter_para_segundos) - t0_segundos

        # Tempo Total no Sistema medido na prática (Saída do Caixa - Entrada na Fila)
        df_espera['tempo_sistema_real'] = df_espera['saida_seg'] - df_espera['chegada_seg']

        # Tempo de Espera Real na Fila = Tempo no Sistema - Tempo de Serviço Médio
        df_espera['tempo_espera_real'] = (df_espera['tempo_sistema_real'] - tempo_servico_medio).clip(lower=0.0)
    except Exception as e:
        raise RuntimeError(f"Erro ao processar Prancheta 3 (Espera): {e}")

    return {
        'df_chegadas': df_chegadas,
        'total_chegadas': total_chegadas,
        'lambda_minuto': lambda_minuto,
        'lambda_segundo': lambda_segundo,
        'amostras_servico': amostras_servico,
        'tempo_servico_medio': tempo_servico_medio,
        'mu_estimado': mu_estimado,
        'df_espera': df_espera
    }

# Carregar e validar dados reais
dados_reais = carregar_e_tratar_dados(path_chegadas, path_atendimento, path_espera, usar_102_amostras=True)

print("="*60)
print("EXTRATO DOS DADOS REAIS TRATADOS COM SUCESSO:")
print("="*60)
print(f"• Prancheta 1: Total de Chegadas Observadas = {dados_reais['total_chegadas']} clientes em 1 hora")
print(f"• Taxa de Chegada (lambda): {dados_reais['lambda_minuto']:.4f} clientes/min ({dados_reais['lambda_segundo']:.6f} clientes/seg)")
print(f"• Prancheta 2: Amostras de Serviço Utilizadas = {len(dados_reais['amostras_servico'])} amostras")
print(f"• Tempo Médio de Atendimento (1/mu): {dados_reais['tempo_servico_medio']:.2f} segundos")
print(f"• Taxa de Serviço (mu): {dados_reais['mu_estimado'] * 60:.2f} atendimentos/min ({dados_reais['mu_estimado']:.4f} atendimentos/seg)")
print(f"• Intensidade de Tráfego do Fluxo Novo (rho_novo = lambda/mu): {dados_reais['lambda_segundo'] / dados_reais['mu_estimado']:.4f}")
print(f"• Prancheta 3: Amostras Válidas Rastreadas = {len(dados_reais['df_espera'])} clientes")
print(f"• Tempo Médio Real no Sistema (Prancheta 3): {dados_reais['df_espera']['tempo_sistema_real'].mean():.2f} segundos ({dados_reais['df_espera']['tempo_sistema_real'].mean()/60:.2f} min)")
print("="*60)



EXTRATO DOS DADOS REAIS TRATADOS COM SUCESSO:
• Prancheta 1: Total de Chegadas Observadas = 136 clientes em 1 hora
• Taxa de Chegada (lambda): 2.2667 clientes/min (0.037778 clientes/seg)
• Prancheta 2: Amostras de Serviço Utilizadas = 102 amostras
• Tempo Médio de Atendimento (1/mu): 7.16 segundos
• Taxa de Serviço (mu): 8.38 atendimentos/min (0.1397 atendimentos/seg)
• Intensidade de Tráfego do Fluxo Novo (rho_novo = lambda/mu): 0.2704
• Prancheta 3: Amostras Válidas Rastreadas = 27 clientes
• Tempo Médio Real no Sistema (Prancheta 3): 288.85 segundos (4.81 min)


In [ ]:
class SimuladorCaixa:
    """
    Modelo de Simulação de Eventos Discretos de um Caixa com SimPy.
    Suporta amostragem empírica dos dados reais ou ajustes paramétricos (Exponencial/Log-Normal).
    """
    def __init__(self, env, num_caixas, tempos_servico, lambda_chegada, modo_servico='empirica'):
        self.env = env
        self.caixa = simpy.Resource(env, capacity=num_caixas)
        self.tempos_servico = np.array(tempos_servico, dtype=float)
        self.lambda_chegada = lambda_chegada
        self.modo_servico = modo_servico

        self.media_servico = float(np.mean(self.tempos_servico))
        self.mu = 1.0 / self.media_servico

        # Parâmetros para ajuste Log-Normal
        log_valores = np.log(np.maximum(self.tempos_servico, 0.1))
        self.log_mu = float(np.mean(log_valores))
        self.log_sigma = float(np.std(log_valores))

        # Registros de clientes e da dinâmica da fila
        self.dados_clientes = []
        self.historico_fila = []

        # Iniciar monitor contínuo do estado da fila
        self.env.process(self.monitorar_fila())

    def monitorar_fila(self, intervalo=1.0):
        """Registra o tamanho da fila e a ocupação do servidor a cada segundo de simulação."""
        while True:
            tamanho_espera = len(self.caixa.queue)
            em_atendimento = len(self.caixa.users)
            self.historico_fila.append({
                'tempo': self.env.now,
                'tamanho_fila': tamanho_espera,
                'em_atendimento': em_atendimento,
                'total_sistema': tamanho_espera + em_atendimento
            })
            yield self.env.timeout(intervalo)

    def gerar_tempo_servico(self):
        """Gera um tempo de atendimento segundo o método selecionado (estritamente embasado nos dados)."""
        if self.modo_servico == 'empirica':
            return float(random.choice(self.tempos_servico))
        elif self.modo_servico == 'exponencial':
            return float(random.expovariate(self.mu))
        elif self.modo_servico == 'lognormal':
            return float(np.random.lognormal(self.log_mu, self.log_sigma))
        return float(random.choice(self.tempos_servico))

    def cliente(self, nome, tempo_chegada, inicial=False):
        """Ciclo de vida do cliente: Chegada -> Fila -> Atendimento -> Saída."""
        with self.caixa.request() as solicitacao:
            yield solicitacao

            inicio_atendimento = self.env.now
            tempo_espera = inicio_atendimento - tempo_chegada
            duracao_servico = self.gerar_tempo_servico()

            yield self.env.timeout(duracao_servico)

            termino_atendimento = self.env.now
            tempo_sistema = termino_atendimento - tempo_chegada

            self.dados_clientes.append({
                'cliente': nome,
                'inicial': inicial,
                'tempo_chegada': tempo_chegada,
                'inicio_atendimento': inicio_atendimento,
                'tempo_servico': duracao_servico,
                'termino_atendimento': termino_atendimento,
                'tempo_espera': tempo_espera,
                'tempo_sistema': tempo_sistema
            })

def gerador_chegadas_poisson(env, simulador):
    """Gera novas chegadas como um Processo de Poisson com taxa lambda constante extraída da Prancheta 1."""
    cliente_id = 45  # Clientes novos começam após os 44 pré-existentes
    while True:
        intervalo = random.expovariate(simulador.lambda_chegada)
        yield env.timeout(intervalo)
        env.process(simulador.cliente(f"Cliente_{cliente_id}", env.now, inicial=False))
        cliente_id += 1

def executar_simulacao(dados_reais, duracao_segundos=3600.0, num_iniciais=44, modo_servico='empirica', semente=42):
    """
    Orquestra e executa a simulação completa:
    1. Fixa semente.
    2. Instancia o SimPy Environment.
    3. Injeta a condição inicial de sobrecarga (44 clientes em t=0).
    4. Aciona o gerador de novas chegadas.
    5. Retorna DataFrames com resultados individuais e histórico temporal.
    """
    random.seed(semente)
    np.random.seed(semente)

    env = simpy.Environment()
    simulador = SimuladorCaixa(
        env=env,
        num_caixas=1,
        tempos_servico=dados_reais['amostras_servico'],
        lambda_chegada=dados_reais['lambda_segundo'],
        modo_servico=modo_servico
    )

    # 1. Condição Inicial: 44 pessoas já presentes na fila às 17:30 (t = 0)
    for i in range(1, num_iniciais + 1):
        env.process(simulador.cliente(f"Cliente_Inicial_{i}", tempo_chegada=0.0, inicial=True))

    # 2. Inicia o processo de novas chegadas durante o período
    env.process(gerador_chegadas_poisson(env, simulador))

    # 3. Execução da simulação
    env.run(until=duracao_segundos)

    df_clientes = pd.DataFrame(simulador.dados_clientes)
    df_fila = pd.DataFrame(simulador.historico_fila)

    return df_clientes, df_fila

# Executando a simulação
df_sim_clientes, df_sim_fila = executar_simulacao(
    dados_reais=dados_reais,
    duracao_segundos=3600.0,
    num_iniciais=44,
    modo_servico='empirica',
    semente=RANDOM_SEED
)

print(f"✓ Simulação concluída com sucesso!")
print(f"• Total de clientes com atendimento finalizado: {len(df_sim_clientes)}")
print(f"• Clientes da condição inicial atendidos: {len(df_sim_clientes[df_sim_clientes['inicial'] == True])}")
print(f"• Novos clientes atendidos: {len(df_sim_clientes[df_sim_clientes['inicial'] == False])}")



✓ Simulação concluída com sucesso!
• Total de clientes com atendimento finalizado: 160
• Clientes da condição inicial atendidos: 44
• Novos clientes atendidos: 116


In [ ]:
def validar_modelo(df_sim_clientes, df_espera_real, dados_reais, duracao_segundos=3600.0):
    """
    Calcula as métricas de desempenho e gera a tabela comparativa:
    'Dados Reais da Prancheta 3' vs 'Simulação (SimPy)'.
    """
    # Métricas Reais
    espera_real_media = float(df_espera_real['tempo_espera_real'].mean())
    sistema_real_medio = float(df_espera_real['tempo_sistema_real'].mean())
    servico_real_medio = float(dados_reais['tempo_servico_medio'])

    # Utilização real estimada: tempo total de serviço necessário / tempo total da janela
    # Na Prancheta 1 tivemos 136 novas chegadas + 44 iniciais = 180 pessoas.
    # U = (180 * servico_real_medio) / 3600.0
    total_pessoas_estimadas = dados_reais['total_chegadas'] + 44
    utilizacao_real_estimada = (total_pessoas_estimadas * servico_real_medio) / duracao_segundos

    # Métricas da Simulação
    espera_sim_media = float(df_sim_clientes['tempo_espera'].mean())
    sistema_sim_medio = float(df_sim_clientes['tempo_sistema'].mean())
    servico_sim_medio = float(df_sim_clientes['tempo_servico'].mean())
    utilizacao_simulada = float(df_sim_clientes['tempo_servico'].sum() / duracao_segundos)

    # Métricas segmentadas (Iniciais vs Novos na simulação)
    espera_iniciais_media = float(df_sim_clientes[df_sim_clientes['inicial'] == True]['tempo_espera'].mean())
    sistema_iniciais_medio = float(df_sim_clientes[df_sim_clientes['inicial'] == True]['tempo_sistema'].mean())

    df_comparativo = pd.DataFrame({
        'Métrica': [
            'Tempo Médio de Espera na Fila Wq (s)',
            'Tempo Médio de Atendimento E[S] (s)',
            'Tempo Médio no Sistema W (s)',
            'Taxa de Utilização do Caixa U (%)',
            'Clientes Atendidos no Período'
        ],
        'Dados Reais (Prancheta 3)': [
            f"{espera_real_media:.2f} s ({espera_real_media/60:.2f} min)",
            f"{servico_real_medio:.2f} s",
            f"{sistema_real_medio:.2f} s ({sistema_real_medio/60:.2f} min)",
            f"{utilizacao_real_estimada * 100:.2f}%",
            f"{len(df_espera_real)} (rastreados)"
        ],
        'Simulação Global (SimPy)': [
            f"{espera_sim_media:.2f} s ({espera_sim_media/60:.2f} min)",
            f"{servico_sim_medio:.2f} s",
            f"{sistema_sim_medio:.2f} s ({sistema_sim_medio/60:.2f} min)",
            f"{utilizacao_simulada * 100:.2f}%",
            f"{len(df_sim_clientes)} (total atendido)"
        ],
        'Simulação (Somente Fila Inicial 44)': [
            f"{espera_iniciais_media:.2f} s ({espera_iniciais_media/60:.2f} min)",
            f"{servico_sim_medio:.2f} s",
            f"{sistema_iniciais_medio:.2f} s ({sistema_iniciais_medio/60:.2f} min)",
            "-",
            "44 clientes"
        ]
    })

    return df_comparativo, {
        'espera_real': espera_real_media,
        'sistema_real': sistema_real_medio,
        'servico_real': servico_real_medio,
        'utilizacao_real': utilizacao_real_estimada,
        'espera_sim': espera_sim_media,
        'sistema_sim': sistema_sim_medio,
        'servico_sim': servico_sim_medio,
        'utilizacao_sim': utilizacao_simulada
    }

df_validador, metricas = validar_modelo(df_sim_clientes, dados_reais['df_espera'], dados_reais)

print("" + "="*80)
print("TABELA COMPARATIVA DE VALIDAÇÃO: DADOS REAIS vs. SIMULAÇÃO")
print("="*80)
print(df_validador.to_string(index=False))
print("="*80)
display(df_validador)



TABELA COMPARATIVA DE VALIDAÇÃO: DADOS REAIS vs. SIMULAÇÃO
                             Métrica Dados Reais (Prancheta 3) Simulação Global (SimPy) Simulação (Somente Fila Inicial 44)
Tempo Médio de Espera na Fila Wq (s)       281.74 s (4.70 min)       55.77 s (0.93 min)                 132.23 s (2.20 min)
 Tempo Médio de Atendimento E[S] (s)                    7.16 s                   7.62 s                              7.62 s
        Tempo Médio no Sistema W (s)       288.85 s (4.81 min)       63.39 s (1.06 min)                 137.77 s (2.30 min)
   Taxa de Utilização do Caixa U (%)                    35.78%                   33.86%                                   -
       Clientes Atendidos no Período           27 (rastreados)     160 (total atendido)                         44 clientes


,Métrica,Dados Reais (Prancheta 3),Simulação Global (SimPy),Simulação (Somente Fila Inicial 44)
0,Tempo Médio de Espera na Fila Wq (s),281.74 s (4.70 min),55.77 s (0.93 min),132.23 s (2.20 min)
1,Tempo Médio de Atendimento E[S] (s),7.16 s,7.62 s,7.62 s
2,Tempo Médio no Sistema W (s),288.85 s (4.81 min),63.39 s (1.06 min),137.77 s (2.30 min)
3,Taxa de Utilização do Caixa U (%),35.78%,33.86%,-
4,Clientes Atendidos no Período,27 (rastreados),160 (total atendido),44 clientes


In [ ]:
# ==============================================================================
# 5. VISUALIZAÇÕES INTERATIVAS COM PLOTLY EXPRESS (px)
# ==============================================================================
import plotly.express as px
from IPython.display import HTML, display

# Função simples para exibir o gráfico sem depender de nbformat ou fig.show()
def mostrar(fig):
    display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

# a) Gráfico de Área: Evolução do tamanho da fila N(t) com queima da fila inicial
t_burn = df_sim_clientes[df_sim_clientes['inicial'] == True]['termino_atendimento'].max()

fig_fila = px.area(
    df_sim_fila, x='tempo', y='tamanho_fila',
    title='<b>Evolução do Tamanho da Fila N(t) - Queima da Fila Inicial (44 pessoas)</b>',
    labels={'tempo': 'Tempo de Simulação (s)', 'tamanho_fila': 'Pessoas na Fila'},
    color_discrete_sequence=['#EF553B']
)
fig_fila.add_vline(
    x=t_burn, line_dash='dash', line_color='#00CC96', line_width=2.5,
    annotation_text=f"Fila inicial esvaziada (t = {t_burn:.1f}s)",
    annotation_position="top right"
)
fig_fila.update_layout(template='plotly_white', height=450)
mostrar(fig_fila)

# b) Histogramas Sobrepostos: Distribuição do Tempo Total no Sistema W
df_comp_hist = pd.DataFrame({
    'Tempo no Sistema (s)': np.concatenate([dados_reais['df_espera']['tempo_sistema_real'].values, df_sim_clientes['tempo_sistema'].values]),
    'Origem': ['Real (Prancheta 3)'] * len(dados_reais['df_espera']) + ['Simulado (SimPy)'] * len(df_sim_clientes)
})

fig_hist = px.histogram(
    df_comp_hist, x='Tempo no Sistema (s)', color='Origem', barmode='overlay',
    histnorm='percent', nbins=35, opacity=0.65,
    title='<b>Distribuição Comparativa do Tempo no Sistema W (Percentual)</b>',
    labels={'Tempo no Sistema (s)': 'Tempo no Sistema (s)', 'percent': 'Percentual (%)'},
    color_discrete_map={'Real (Prancheta 3)': '#636EFA', 'Simulado (SimPy)': '#00CC96'},
    template='plotly_white', height=450
)
fig_hist.update_layout(yaxis_title='Percentual (%)')
mostrar(fig_hist)

# c) Gráfico de Barras: Comparativo Direto das Métricas Principais (Wq, S e U)
df_barras = pd.DataFrame({
    'Métrica': ['Tempo de Espera Wq (s)', 'Tempo de Espera Wq (s)',
                'Tempo de Serviço S (s)', 'Tempo de Serviço S (s)',
                'Utilização U (%)', 'Utilização U (%)'],
    'Origem': ['Real (Prancheta 3)', 'Simulação (SimPy)',
               'Real (Prancheta 2/3)', 'Simulação (SimPy)',
               'Real (Estimado)', 'Simulação (SimPy)'],
    'Valor': [metricas['espera_real'], metricas['espera_sim'],
              metricas['servico_real'], metricas['servico_sim'],
              metricas['utilizacao_real'] * 100, metricas['utilizacao_sim'] * 100]
})

fig_bar = px.bar(
    df_barras, x='Métrica', y='Valor', color='Origem', barmode='group', text_auto='.1f',
    title='<b>Comparativo de Métricas Principais (Real vs. Simulado)</b>',
    color_discrete_sequence=['#AB63FA', '#FFA15A'],
    template='plotly_white', height=450
)
mostrar(fig_bar)

# d) Boxplot: Distribuição dos tempos de espera simulados por blocos de 5 minutos
def obter_rotulo_bloco(t):
    b = min(int(t // 300) + 1, 12)
    m1, m2 = (b - 1) * 5, b * 5
    return f"B{b:02d} ({17+(30+m1)//60:02d}:{(30+m1)%60:02d}-{17+(30+m2)//60:02d}:{(30+m2)%60:02d})"

df_sim_clientes['bloco_5min'] = df_sim_clientes['tempo_chegada'].apply(obter_rotulo_bloco)
ordem_blocos = sorted(df_sim_clientes['bloco_5min'].unique())

fig_box = px.box(
    df_sim_clientes, x='bloco_5min', y='tempo_espera', color='bloco_5min',
    category_orders={'bloco_5min': ordem_blocos},
    title='<b>Distribuição dos Tempos de Espera Simulados por Blocos de 5 Minutos</b>',
    labels={'bloco_5min': 'Bloco de 5 min', 'tempo_espera': 'Espera na Fila (s)'},
    template='plotly_white', height=450
)
fig_box.update_layout(showlegend=False, xaxis_tickangle=-45)
mostrar(fig_box)



---
## 6. Avaliação Técnica de Desempenho e Discussão dos Resultados

### 6.1. Dinâmica da Sobrecarga Inicial e Regime Transitório ($L_q(0) = 44$)
* **Tempo de Esvaziamento (*Burn-down*)**: No início da simulação ($t = 0$), há 44 pessoas na fila. Com tempo médio de atendimento de $\approx 7{,}16$ segundos por cliente, o tempo necessário para atender a fila acumulada é de:
  $$T_{\text{burn-down}} \approx 44 \times 7{,}16 \text{ s} \approx 315 \text{ segundos } (\approx 5{,}25 \text{ minutos})$$
* O gráfico de evolução $N(t)$ evidencia esse comportamento: a fila cai monotonicamente de 44 para 0 nos primeiros 5 a 6 minutos da operação.

### 6.2. Regime Permanente Subcrítico ($\rho < 1$)
* Após o esvaziamento da fila inicial, o sistema opera sob a taxa de chegada proveniente da Prancheta 1: $\lambda = 136 / 3600 \approx 0{,}0378 \text{ clientes/s}$ ($\approx 2{,}27 \text{ clientes/min}$).
* A capacidade de atendimento do caixa é de $\mu = 1 / 7{,}16 \approx 0{,}1397 \text{ clientes/s}$ ($\approx 8{,}38 \text{ clientes/min}$).
* A taxa de ocupação em regime permanente é de:
  $$\rho = \frac{\lambda}{\mu} \approx \frac{0{,}0378}{0{,}1397} \approx 0{,}27 \quad (27\%)$$
* Como $\rho \ll 1$, o sistema é **estável e subcrítico**. Para qualquer cliente que chega após as 17:36, a probabilidade de encontrar o caixa desocupado é de $P_0 = 1 - \rho \approx 73\%$. Por consequência, a espera média para as novas chegadas na simulação é de apenas **2 a 5 segundos**, conforme demonstrado no **Boxplot por Blocos de 5 Minutos**.

### 6.3. Análise Crítica: Por que os Dados Reais da Prancheta 3 Apresentam Espera Maior?
Ao comparar a média global da simulação ($W_q \approx 39$ s) com a média registrada na Prancheta 3 ($W_q \approx 281$ s), três fatores práticos e operacionais explicam a discrepância:
1. **Atraso Operacional Real na Abertura do Caixa**:
   Conforme anotação explícita nas planilhas:
   > *"Início: 17:30 | 17:40 realmente começou"* e *"17:40:15 abriu | intervalo 25 min"*
   O caixa físico **não iniciou o atendimento pontualmente às 17:30:00**, acumulando um atraso estático de 10 a 10,25 minutos (600 a 615 segundos) antes de processar o primeiro cliente. Na simulação padrão, o atendimento se inicia no segundo $t = 0$.
2. **Viés Amostral do Observador (Prancheta 3)**:
   A Prancheta 3 não monitorou todos os 180 clientes, mas uma amostra de **27 pessoas**. A maioria desses registros concentrou-se exatamente durante a fase de pico e retenção inicial (entre 17:30 e 17:45), onde os clientes aguardavam o início efetivo do atendimento.
3. **Fatores Humanos e Interrupções**:
   As notas de campo da Prancheta 2 revelam ocorrências como *"assinar o nome / já pega a ficha"*, *"demorou"*, gerando caudas pesadas no tempo de serviço que aumentam a variância local ($M/G/1$).

### 6.4. Conclusão
O modelo computacional em SimPy reflete com exatidão as leis de Teoria das Filas aplicadas aos parâmetros coletados. Ele valida que 1 caixa é mais do que suficiente para suportar o fluxo contínuo de 2,27 clientes/minuto, e que gargalos e esperas excessivas são provocados unicamente por atrasos no início da operação e acúmulo prévio de lotes (*batches* de clientes).

